# Regresi Linier — Dataset Titik A sampai G

Notebook ini menghitung koefisien regresi linier menggunakan dua metode:
1. **sklearn** `LinearRegression`
2. **Analitik manual** dengan rumus Normal Equation: **β̂ = (XᵀX)⁻¹ XᵀY**

Dataset: 7 titik koordinat dari GeoGebra (A–G)

| Titik | x | y |
|-------|---|---|
| A | 2 | 2 |
| B | 4 | 3 |
| C | 3 | 5 |
| D | 3 | 4 |
| E | 3 | 3 |
| F | 4 | 5 |
| G | 5 | 6 |

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## 2. Input Dataset

In [ ]:
# Dataset titik A sampai G dari GeoGebra
labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
X_raw  = np.array([2, 4, 3, 3, 3, 4, 5], dtype=float)   # variabel bebas (x)
Y      = np.array([2, 3, 5, 4, 3, 5, 6], dtype=float)   # variabel terikat (y)

df = pd.DataFrame({'Titik': labels, 'x': X_raw, 'y': Y})
df = df.set_index('Titik')
print(df.to_string())

## 3. Metode 1 — sklearn LinearRegression

sklearn memerlukan input X berbentuk 2D array (matriks kolom).

In [ ]:
X_2d = X_raw.reshape(-1, 1)   # ubah jadi shape (7, 1)

model = LinearRegression()
model.fit(X_2d, Y)

b0_sk = model.intercept_
b1_sk = model.coef_[0]

Y_pred_sk = model.predict(X_2d)

print('=' * 50)
print('METODE 1 — sklearn LinearRegression')
print('=' * 50)
print(f'b0 (intercept)  : {b0_sk:.6f}')
print(f'b1 (koefisien)  : {b1_sk:.6f}')
print(f'Persamaan       : y = {b1_sk:.4f}x + {b0_sk:.4f}')
print(f'R² Score        : {r2_score(Y, Y_pred_sk):.4f}')
print(f'MSE             : {mean_squared_error(Y, Y_pred_sk):.4f}')

## 4. Metode 2 — Analitik Manual: β̂ = (XᵀX)⁻¹ XᵀY

### Step 1: Susun Matriks X dengan Kolom Dummy

Kolom pertama berisi **1 semua** → placeholder untuk koefisien b0  
Kolom kedua berisi nilai prediktor x

In [ ]:
ones  = np.ones((len(X_raw), 1))          # kolom dummy
X_mat = np.column_stack([ones, X_raw])     # matriks X: shape (7, 2)

print('Matriks X (kolom 1 = dummy, kolom 2 = prediktor):')
print(X_mat)
print(f'\nShape X: {X_mat.shape}  →  {X_mat.shape[0]} baris, {X_mat.shape[1]} kolom')

print('\nVektor Y:')
print(Y)

### Step 2: Hitung XᵀX

**Transpose** = balik baris jadi kolom. Lalu kalikan Xᵀ dengan X → hasilkan matriks **2×2** (square matrix).

In [ ]:
Xt  = X_mat.T      # Transpose X: shape (2, 7)
XtX = Xt @ X_mat   # perkalian matriks: (2,7) × (7,2) = (2,2)

print('Xᵀ (Transpose X):')
print(Xt)
print()
print('XᵀX:')
print(XtX)
print()
print(f'  XᵀX[0,0] = jumlah kolom dummy × dummy = {int(XtX[0,0])}')
print(f'  XᵀX[0,1] = jumlah x = {int(XtX[0,1])}')
print(f'  XᵀX[1,1] = jumlah x² = {int(XtX[1,1])}')

### Step 3: Invers (XᵀX)⁻¹

Hanya **matriks persegi** yang dapat di-invers.  
Untuk matriks 2×2: swap diagonal → bagi determinan.

In [ ]:
# Hitung determinan secara manual
a, b = XtX[0, 0], XtX[0, 1]
c, d = XtX[1, 0], XtX[1, 1]

det = a * d - b * c
print(f'Determinan = ({a:.0f} × {d:.0f}) − ({b:.0f} × {c:.0f})')
print(f'           = {a*d:.0f} − {b*c:.0f}')
print(f'           = {det:.0f}')
print()

# Invers manual 2×2: (1/det) × [[d, -b], [-c, a]]
XtX_inv_manual = (1 / det) * np.array([[d, -b], [-c, a]])
print('(XᵀX)⁻¹ — perhitungan manual:')
print(XtX_inv_manual)
print()

# Verifikasi dengan numpy
XtX_inv = np.linalg.inv(XtX)
print('(XᵀX)⁻¹ — numpy.linalg.inv (verifikasi):')
print(XtX_inv)
print()
match = np.allclose(XtX_inv_manual, XtX_inv)
print(f'Manual vs numpy: {"✓ IDENTIK" if match else "✗ BERBEDA"}')

### Step 4: Hitung XᵀY

In [ ]:
XtY = Xt @ Y   # (2,7) × (7,) = (2,)

print('XᵀY:')
print(f'  XᵀY[0] = Σ(1 × yi) = {" + ".join(str(int(v)) for v in Y)} = {int(XtY[0])}')
xiyistr = ' + '.join(f'{int(xi)}×{int(yi)}' for xi, yi in zip(X_raw, Y))
print(f'  XᵀY[1] = Σ(xi × yi) = {xiyistr} = {int(XtY[1])}')
print()
print(f'XᵀY = {XtY}')

### Step 5: Hitung β̂ = (XᵀX)⁻¹ · XᵀY

In [ ]:
beta = XtX_inv @ XtY   # (2,2) × (2,) = (2,)

b0_an = beta[0]
b1_an = beta[1]

print('=' * 50)
print('METODE 2 — Analitik Normal Equation')
print('=' * 50)
print(f'β̂  = {beta}')
print()
print(f'b0 (intercept)  : {b0_an:.6f}')
print(f'b1 (koefisien)  : {b1_an:.6f}')
print(f'Persamaan       : y = {b1_an:.4f}x + {b0_an:.4f}')

print()
print('Detail perhitungan:')
print(f'  b0 = ({XtX_inv[0,0]:.4f} × {XtY[0]:.0f}) + ({XtX_inv[0,1]:.4f} × {XtY[1]:.0f})')
print(f'     = {XtX_inv[0,0]*XtY[0]:.4f} + ({XtX_inv[0,1]*XtY[1]:.4f}) = {b0_an:.4f}')
print(f'  b1 = ({XtX_inv[1,0]:.4f} × {XtY[0]:.0f}) + ({XtX_inv[1,1]:.4f} × {XtY[1]:.0f})')
print(f'     = {XtX_inv[1,0]*XtY[0]:.4f} + {XtX_inv[1,1]*XtY[1]:.4f} = {b1_an:.4f}')

## 5. Verifikasi — Kedua Metode Harus Sama

In [ ]:
print('=' * 50)
print('VERIFIKASI')
print('=' * 50)
print(f'{"Metode":<15} {"b0":>12} {"b1":>12}')
print('-' * 42)
print(f'{"sklearn":<15} {b0_sk:>12.6f} {b1_sk:>12.6f}')
print(f'{"Analitik":<15} {b0_an:>12.6f} {b1_an:>12.6f}')
print('-' * 42)
ok_b0 = '✓ SAMA' if abs(b0_sk - b0_an) < 1e-9 else '✗ BEDA'
ok_b1 = '✓ SAMA' if abs(b1_sk - b1_an) < 1e-9 else '✗ BEDA'
print(f'{"Status":<15} {ok_b0:>12} {ok_b1:>12}')

## 6. Tabel Prediksi vs Nilai Asli

In [ ]:
Y_pred = b1_an * X_raw + b0_an
residuals = Y - Y_pred

df_result = pd.DataFrame({
    'Titik'       : labels,
    'x'           : X_raw.astype(int),
    'y asli'      : Y.astype(int),
    'ŷ prediksi'  : np.round(Y_pred, 4),
    'Residual'    : np.round(residuals, 4)
})
df_result = df_result.set_index('Titik')
print(df_result.to_string())
print()
print(f'MSE  = {mean_squared_error(Y, Y_pred):.4f}')
print(f'R²   = {r2_score(Y, Y_pred):.4f}')

## 7. Visualisasi

In [ ]:
x_line = np.linspace(1.5, 5.5, 200)
y_line = b1_an * x_line + b0_an

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Plot 1: Scatter + garis regresi + residual ──────────────────
ax = axes[0]
ax.plot(x_line, y_line, color='crimson', linewidth=2,
        label=f'ŷ = {b1_an:.2f}x + {b0_an:.2f}', zorder=3)
ax.scatter(X_raw, Y, color='royalblue', s=90, zorder=5,
           label='Data A–G')
for xi, yi, ypi, lbl in zip(X_raw, Y, Y_pred, labels):
    # garis residual
    ax.plot([xi, xi], [yi, ypi], color='gray',
            linestyle='--', linewidth=1, alpha=0.6)
    # label titik
    ax.annotate(lbl, (xi, yi), textcoords='offset points',
                xytext=(6, 4), fontsize=10, color='navy', fontweight='bold')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Regresi Linier — Titik A sampai G\n(garis putus = residual)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(1, 6)
ax.set_ylim(1, 7)

# ── Plot 2: Tabel hasil ─────────────────────────────────────────
ax2 = axes[1]
ax2.axis('off')

col_labels = ['Metode', 'b0', 'b1', 'Persamaan']
rows_data  = [
    ['sklearn',  f'{b0_sk:.4f}', f'{b1_sk:.4f}', f'y = {b1_sk:.2f}x + {b0_sk:.2f}'],
    ['Analitik', f'{b0_an:.4f}', f'{b1_an:.4f}', f'y = {b1_an:.2f}x + {b0_an:.2f}'],
    ['Status',   '✓ SAMA',       '✓ SAMA',       '✓ IDENTIK'],
]
table = ax2.table(cellText=rows_data, colLabels=col_labels,
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1.3, 2.2)
# warnai header
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')
# warnai baris status
for j in range(len(col_labels)):
    table[3, j].set_facecolor('#E2EFDA')

ax2.set_title(f'Ringkasan Hasil\nR² = {r2_score(Y, Y_pred):.4f}  |  MSE = {mean_squared_error(Y, Y_pred):.4f}',
              fontsize=11, pad=20)

plt.tight_layout()
plt.savefig('plot_regresi_AG.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot tersimpan → plot_regresi_AG.png')

## 8. Rekap Langkah Perhitungan Manual

| Langkah | Operasi | Dimensi | Hasil |
|---------|---------|---------|-------|
| 1 | Susun X + kolom dummy | X: 7×2, Y: 7×1 | Matriks siap dihitung |
| 2 | XᵀX | (2×7) × (7×2) → 2×2 | [[7, 24], [24, 88]] |
| 3 | (XᵀX)⁻¹ | det=40 | [[2.2, -0.6], [-0.6, 0.175]] |
| 4 | XᵀY | (2×7) × (7×1) → 2×1 | [28, 102] |
| 5 | β̂ = (XᵀX)⁻¹ · XᵀY | (2×2) × (2×1) → 2×1 | b0=0.4, b1=1.05 |

**Persamaan akhir: `y = 1.05·x + 0.4`**